In [1]:
# 05_evaluation_airbnb.ipynb - GÜNCELLENMİŞ
# ===============================
# 1. Bibliotheken importieren
# ===============================
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("📊 Evaluation gestartet...")

# ===============================
# 2. Modell und Daten laden
# ===============================
base_path = os.path.dirname(os.getcwd())
model_path = os.path.join(base_path, "models", "airbnb_model.pkl")
data_path = os.path.join(base_path, "data", "processed")

# Modell laden
model = joblib.load(model_path)
print("✅ Modell geladen")

# Testdaten laden
X_test = pd.read_csv(os.path.join(data_path, "X_test.csv"))
y_test = pd.read_csv(os.path.join(data_path, "y_test.csv"))

# Sicherstellen, dass alle Daten numerisch sind
for col in X_test.columns:
    if X_test[col].dtype == 'object':
        X_test[col] = pd.to_numeric(X_test[col], errors='coerce')
X_test = X_test.fillna(0)
y_test = y_test.values.ravel()

print(f"📊 Testdaten: {X_test.shape}")

# ===============================
# 3. Vorhersagen erstellen
# ===============================
y_pred = model.predict(X_test)

# ===============================
# 4. Modellbewertung mit Metriken
# ===============================
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("\n📈 Modellbewertung:")
print(f"MAE  (Mittlerer absoluter Fehler): {mae:.2f}")
print(f"RMSE (Wurzel des mittleren quadratischen Fehlers): {rmse:.2f}")
print(f"R²   (Bestimmtheitsmaß): {r2:.2f}")

# ===============================
# 5. Visualisierungen erstellen
# ===============================
output_path = os.path.join(base_path, "output")
os.makedirs(output_path, exist_ok=True)

# Residuen-Analyse
residuals = y_test - y_pred

plt.figure(figsize=(8,5))
sns.histplot(residuals, kde=True, color="steelblue")
plt.title("Verteilung der Residuen")
plt.xlabel("Residuum (Preis - Vorhersage)")
plt.ylabel("Häufigkeit")
plt.tight_layout()
plt.savefig(os.path.join(output_path, "residuals_distribution.png"))
plt.close()

# Tatsächliche vs. vorhergesagte Werte
plt.figure(figsize=(6,6))
plt.scatter(y_test, y_pred, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color="red", linestyle="--")
plt.xlabel("Tatsächlicher Preis")
plt.ylabel("Vorhergesagter Preis")
plt.title("Tatsächlicher vs. vorhergesagter Preis")
plt.tight_layout()
plt.savefig(os.path.join(output_path, "actual_vs_predicted.png"))
plt.close()

# Feature-Importance
try:
    importances = model.feature_importances_
    feature_names = X_test.columns
    
    feat_imp_df = pd.DataFrame({
        "Merkmal": feature_names,
        "Bedeutung": importances
    }).sort_values(by="Bedeutung", ascending=False)

    plt.figure(figsize=(10,6))
    sns.barplot(data=feat_imp_df, x="Bedeutung", y="Merkmal", palette="viridis")
    plt.title("Feature Importance")
    plt.tight_layout()
    plt.savefig(os.path.join(output_path, "feature_importances.png"))
    plt.close()
    print("✅ Feature-Importance Diagramm erstellt")
    
except AttributeError:
    print("ℹ️ Feature-Importances nicht verfügbar für dieses Modell.")

print("\n🎯 EVALUATION ERFOLGREICH ABGESCHLOSSEN!")

📊 Evaluation gestartet...


✅ Modell geladen
📊 Testdaten: (9718, 6)

📈 Modellbewertung:
MAE  (Mittlerer absoluter Fehler): 50.36
RMSE (Wurzel des mittleren quadratischen Fehlers): 88.51
R²   (Bestimmtheitsmaß): 0.39


C:\Users\okans\AppData\Local\Temp\ipykernel_14428\1446585523.py:98: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=feat_imp_df, x="Bedeutung", y="Merkmal", palette="viridis")


✅ Feature-Importance Diagramm erstellt

🎯 EVALUATION ERFOLGREICH ABGESCHLOSSEN!
